# การพยากรณ์ความผันผวนพอร์ต ETF ด้วย LSTM/Transformer และประโยชน์ของ VIX — Colab runner

Notebook นี้รัน repository ที่ **commit `892dcc7a22e03f83c118004c18f499b30f2e0fbc`** (ตรึงไว้เพื่อให้ทำซ้ำได้) ผลทุกอย่างถูกเขียนลง Google Drive ทันที
หาก runtime หลุด ให้รัน cell ตั้งค่า (ข้อ 0–3) ใหม่แล้วรัน cell ที่ค้างอยู่ซ้ำ ระบบจะทำต่อจากจุดเดิม

**ลำดับ:** 0 ตั้งค่า → 1 ทดสอบความถูกต้อง → 2 สร้าง snapshot จากไฟล์ข้อมูลใน Google Drive และ smoke (train/validation) → 3 formal: data + baselines →
4 train 48 runs → 5 freeze (ตรวจ validation) → 6 เปิด test ครั้งเดียว + backtest + กราฟ + handover → 7 mirror ไป Google Sheets → 8 สร้าง bundle ส่งกลับ

> ใช้ GPU: Runtime → Change runtime type → T4 GPU

## 0. ตั้งค่า: mount Google Drive, ตรวจ GPU, clone repository ที่ commit ที่ตรึงไว้

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

GIT_COMMIT = "892dcc7a22e03f83c118004c18f499b30f2e0fbc"                      # do not change for the formal run
REPO_URL   = "https://github.com/Saelee-Kittipop/Term-Project.git"
DRIVE_ROOT = "/content/drive/MyDrive/vix_term_project"   # data, checkpoints and all outputs
SMOKE_RUN  = "smoke_colab_v1"    # separate names: do not overwrite or mix with the formal_v1 evidence run
FORMAL_RUN = "formal_colab_v1"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"          # deterministic cuBLAS
os.environ["VOLVIX_GIT_COMMIT"] = GIT_COMMIT
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU - training will run on CPU"

In [2]:
REPO_DIR = "/content/Term-Project"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {GIT_COMMIT}
head = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"]).decode().strip()
assert head == GIT_COMMIT, f"checked out {head}, expected {GIT_COMMIT}"
%cd {REPO_DIR}
print("repository pinned at", head)

/content/Term-Project
repository pinned at 35f9e6bd6e9a790e08fb554e8ae714b44a710aab


In [3]:
# torch is preinstalled in Colab (CUDA build) and is NOT reinstalled
!pip -q install "arch>=7.0" "yfinance>=0.2.50" "pandas_market_calendars>=4.4,<5" pyyaml pyarrow
import torch, pandas, numpy, arch, yfinance
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| pandas", pandas.__version__,
      "| numpy", numpy.__version__, "| arch", arch.__version__, "| yfinance", yfinance.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
torch 2.14.1+cu130 cuda True | pandas 2.2.3 | numpy 2.4.6 | arch 9.0.0 | yfinance 0.2.66


## 1. ทดสอบความถูกต้องของโค้ด (ต้องผ่านทั้งหมดก่อนทำขั้นถัดไป)
ทดสอบ target alignment, purge, scaler, baselines, cutoff, การบัญชี backtest และ guard ของ test set ด้วยข้อมูลสังเคราะห์

In [4]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
open(f"{DRIVE_ROOT}/pytest_{GIT_COMMIT[:8]}.log", "w").write(r.stdout + r.stderr)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed - stop here and send the log"

......................................                                   [100%]
=============================== warnings summary ===============================
tests/test_models_training_guards.py::test_smoke_run_cannot_open_test
  /usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:910.)
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
 


## 2. สร้าง raw snapshot จากไฟล์ข้อมูลที่ดาวน์โหลดไว้แล้ว (ไม่ดึงจาก Yahoo ใน Colab)
ใช้ไฟล์ชุดเดียวกับใน `Desktop/Term Project/data` ซึ่งอัปโหลดไว้ที่ `MyDrive/vix_term_project/input_files/` แล้ว:
`volvix_yahoo_chart_{ACWI,SHY,XLK,XLV,GLD,VIX}.json` (Yahoo chart API v8), `VIX_History.csv` (Cboe) และ
`F-F_Research_Data_Factors_daily.csv` (Kenneth French)

cell นี้ตรวจ SHA-256 ของทุกไฟล์เทียบกับไฟล์ที่ใช้ใน run `formal_v1` ก่อน ถ้าไม่ตรงจะหยุด จากนั้น pipeline จะคัดลอกไฟล์ต้นฉบับแบบไม่แก้ไขไปไว้ที่
`data/raw/FILES_<เวลา>/original/` และสร้าง CSV ที่ cleaner ใช้ตามกติกาเดียวกับ yfinance `auto_adjust=True`
ถ้ามี snapshot จากไฟล์ชุดเดียวกันอยู่แล้วใน Drive จะใช้ snapshot เดิม

ถ้าจะใช้ไฟล์ชุดใหม่ ให้ตั้ง `ALLOW_DIFFERENT_FILES = True` ผลที่ได้จะเป็นข้อมูลคนละชุดกับ formal_v1 และห้ามนำผลมาปนกัน


In [ ]:
import glob, hashlib, json, pandas as pd
from pathlib import Path
INPUT_DIR = f"{DRIVE_ROOT}/input_files"
ALLOW_DIFFERENT_FILES = False
# SHA-256 of the files used by the formal_v1 run (snapshot FILES_20261118T215048Z)
EXPECTED_SHA256 = {
    "volvix_yahoo_chart_ACWI.json": "40b0d30f3f58c4c03190eecfc5ad3455efff029d916aa444241ec33b2cb07e35",
    "volvix_yahoo_chart_SHY.json":  "e9dfa5532372b742f137b35349766a95c28dc31d25a4ffe2529e887cdab0a0bd",
    "volvix_yahoo_chart_XLK.json":  "19848462dba50e86e4ea69eb66a577c9e201db211df4b8c84a7e904947e9e07e",
    "volvix_yahoo_chart_XLV.json":  "62474c0656e95c40131e0ed5e23a47051dd8bc78566d2bc6bb971a4da2d5f185",
    "volvix_yahoo_chart_GLD.json":  "2d59be73a68324ceae422c8abab112214f658b95d845e060df9bba201c63be1a",
    "volvix_yahoo_chart_VIX.json":  "43dc24a37c757a310cfdacefba4541dbbbb4a0b368847e0204ea06f7b905fc64",
    "VIX_History.csv":              "6a80de0d46497181c18b11ea044aff7f7070d1b900b0b339049d8713dcb5a0cd",
    "F-F_Research_Data_Factors_daily.csv": "3f7910502dd6f188748210e5145d95642de1d8cff56c2cf0fa85985299610ff0",
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

missing = [n for n in EXPECTED_SHA256 if not os.path.exists(f"{INPUT_DIR}/{n}")]
assert not missing, f"missing in {INPUT_DIR}: {missing} - upload them from Desktop/Term Project/data (names must be exact)"
actual = {n: sha256(f"{INPUT_DIR}/{n}") for n in EXPECTED_SHA256}
for n in EXPECTED_SHA256:
    print(("OK      " if actual[n] == EXPECTED_SHA256[n] else "DIFFERS ") + n, actual[n][:16])
if not ALLOW_DIFFERENT_FILES:
    assert actual == EXPECTED_SHA256, "files differ from the formal_v1 inputs - fix the upload or set ALLOW_DIFFERENT_FILES = True"

# reuse an existing FILES_ snapshot built from exactly these files; otherwise build a new one in the smoke data stage
SNAPSHOT_ARG = f"files:{INPUT_DIR}"
for mp in sorted(glob.glob(f"{DRIVE_ROOT}/data/raw/FILES_*/snapshot_metadata.json")):
    files = json.load(open(mp))["files"]
    if all(files.get(f"original/{n}", {}).get("sha256") == h for n, h in actual.items()):
        SNAPSHOT_ARG = Path(mp).parent.name
print("snapshot argument:", SNAPSHOT_ARG)


### Smoke check บน train/validation
Smoke ใช้ 1 config, 1 seed, 2 epochs เพื่อตรวจว่าทุกขั้นรันได้ **ผล smoke ไม่ใช่ผลการทดลอง** และ smoke ไม่สามารถเปิด test ได้

In [ ]:
!python scripts/run_pipeline.py --root {DRIVE_ROOT} --run-name {SMOKE_RUN} --override configs/smoke.yaml \
    --snapshot {SNAPSHOT_ARG} --stages data,baselines,train,freeze,figures,report 2>&1 | tee {DRIVE_ROOT}/smoke.log

In [ ]:
from IPython.display import Markdown, display
snap_id = json.load(open(f"{DRIVE_ROOT}/runs/{SMOKE_RUN}/run_manifest.json"))["data"]["snapshot_id"]
assert snap_id.startswith("FILES_"), f"smoke used {snap_id}, expected a FILES_ snapshot"
meta = json.load(open(f"{DRIVE_ROOT}/data/raw/{snap_id}/snapshot_metadata.json"))
assert all(meta["files"][f"original/{n}"]["sha256"] == h for n, h in actual.items()), "snapshot originals do not match the input files"
SNAPSHOT_ID = snap_id
print("snapshot used:", snap_id, "| provenance:", meta.get("provenance"))
split = pd.read_csv(f"{DRIVE_ROOT}/runs/{SMOKE_RUN}/dataset/split_manifest.csv")
display(split)
display(pd.read_csv(f"{DRIVE_ROOT}/runs/{SMOKE_RUN}/dataset/alignment_checks.csv"))
# same files + same code should give the same samples as formal_v1 (train 2,394 / validation 498 / test 1,503)
same = split.set_index("split")["net_samples"].to_dict() == {"train": 2394, "validation": 498, "test": 1503}
print("sample counts match formal_v1:", same)
display(Markdown(open(f"{DRIVE_ROOT}/runs/{SMOKE_RUN}/reports/data_quality_report_th.md").read()))


## 3. Formal run: เตรียมข้อมูลจาก snapshot เดียวกับ smoke และคำนวณ baselines

In [ ]:
!python scripts/run_pipeline.py --root {DRIVE_ROOT} --run-name {FORMAL_RUN} --snapshot {snap_id} \
    --stages data,baselines 2>&1 | tee -a {DRIVE_ROOT}/formal.log
print(json.dumps(json.load(open(f"{DRIVE_ROOT}/runs/{FORMAL_RUN}/baselines/baseline_info.json"))["garch"], indent=1)[:1500])

## 4. ฝึก 48 runs (resumable)
หาก Colab หลุด ให้รันข้อ 0 ใหม่ แล้วรัน cell นี้ซ้ำ run ที่เสร็จแล้วจะถูกข้าม run ที่ค้างจะทำต่อจาก checkpoint ล่าสุด

In [ ]:
!python scripts/run_pipeline.py --root {DRIVE_ROOT} --run-name {FORMAL_RUN} --stages train 2>&1 | tee -a {DRIVE_ROOT}/formal.log
g = pd.read_csv(f"{DRIVE_ROOT}/runs/{FORMAL_RUN}/training/grid_summary.csv")
print(g["status"].value_counts())

## 5. Freeze: เลือก config, คำนวณ τ, cutoffs และ static comparator จาก train/validation แล้วตรึงพร้อม hash
**หลังจากนี้ห้ามแก้ไข config, ข้อมูล, โมเดล หรือ cutoff** ตรวจตาราง validation ด้านล่างก่อนไปข้อ 6

In [ ]:
!python scripts/run_pipeline.py --root {DRIVE_ROOT} --run-name {FORMAL_RUN} --stages freeze 2>&1 | tee -a {DRIVE_ROOT}/formal.log
fz = json.load(open(f"{DRIVE_ROOT}/runs/{FORMAL_RUN}/frozen/freeze.json"))
print("tau =", fz["tau"]); print(json.dumps(fz["selection"], indent=1)); print(json.dumps(fz["cutoffs"], indent=1))
display(pd.read_csv(f"{DRIVE_ROOT}/runs/{FORMAL_RUN}/validation/validation_metrics.csv"))

## 6. เปิด test ครั้งเดียว → backtest → กราฟ → เอกสาร handover
ระบบตรวจ hash ของทุกสิ่งที่ตรึงไว้ก่อนเปิด test และบันทึก `TEST_OPENED.json` การรันซ้ำต้องใช้ `--allow-test-rerun-reason` (เฉพาะแก้ข้อผิดพลาด implementation ไม่ใช่เพื่อปรับผล)

In [ ]:
!python scripts/run_pipeline.py --root {DRIVE_ROOT} --run-name {FORMAL_RUN} --stages test,backtest,figures,report 2>&1 | tee -a {DRIVE_ROOT}/formal.log
display(pd.read_csv(f"{DRIVE_ROOT}/runs/{FORMAL_RUN}/test/test_metrics.csv"))
display(Markdown(open(f"{DRIVE_ROOT}/runs/{FORMAL_RUN}/reports/handover_th.md").read()))

## 7. Mirror ตารางสรุปไป Google Sheets (ไฟล์ CSV/JSON บน Drive ยังเป็นผลหลัก)

In [ ]:
from google.colab import auth
auth.authenticate_user()
!pip -q install gspread
SPREADSHEET_ID = "15CUiD2KzgJriNChW7yKtuZCbVqTN7tRq05nNRnP3O28"   # mirror sheet in MyDrive/vix_term_project; set "" to create a new one
args = f"--spreadsheet-id {SPREADSHEET_ID}" if SPREADSHEET_ID else ""
!python scripts/mirror_to_sheets.py --run-dir {DRIVE_ROOT}/runs/{FORMAL_RUN} {args}

## 8. สร้าง bundle สำหรับส่งกลับ (ไม่รวม checkpoints/parquet ซึ่งเก็บอยู่ใน Drive)

In [ ]:
!python scripts/make_bundle.py --root {DRIVE_ROOT} --run-name {FORMAL_RUN} --extra-run {SMOKE_RUN}
from google.colab import files
files.download(f"{DRIVE_ROOT}/bundle_{FORMAL_RUN}.zip")